In [141]:
!pip install emoji pyarabic nltk

In [142]:
## Preprocessing and Cleaning:
import nltk
import emoji
import pandas as pd
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize

## Text Representation:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


nltk.download('punkt')        # ---------> 'punkt' is a pre-trained NLTK tokenizer model that enables correct splitting of text into words or sentences.
nltk.download('stopwords')    # ---------> 'stopwords' is an NLTK resource that provides lists of common words (like “the”, “and”) to filter out during text preprocessing.
nltk.download('punkt_tab')    # ---------> 'punkt_tab' is a pre-trained NLTK tokenizer model that allows splitting text into words or sentences for tokenization.

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [143]:
df = pd.read_csv('AAFAQ_Dataset.csv')
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


### Emojies

In [144]:
# checking for emojis in the dataset
emoji.emoji_count(df['QuestionText'] + df['Answer'])

0

there are no emojies in the dataset, so we can skip this step.

### Tashkeel

In [145]:
# Diacritics Removal (stripping tashkeel)
df['QuestionText'] = df['QuestionText'].apply(lambda x: araby.strip_tashkeel(x))
df['Answer'] = df['Answer'].apply(lambda x: araby.strip_tashkeel(x))

df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا من النيتروجين؟,التعليم,الهواء يتكون أساسا من النيتروجين.


### Tatweel

In [146]:
def has_tatweel(text):
    return araby.TATWEEL in text

print("Count of Tatweel:")
print(f"num of tatweel in [QuestionText] = {df['QuestionText'].apply(has_tatweel).sum()}")
print(f"num of tatweel in [Answer] = {df['Answer'].apply(has_tatweel).sum()}")


Count of Tatweel:
num of tatweel in [QuestionText] = 6
num of tatweel in [Answer] = 33


In [147]:
# removing tatweel
df['QuestionText'] = df['QuestionText'].apply(lambda x: araby.strip_tatweel(x))
df['Answer'] = df['Answer'].apply(lambda x: araby.strip_tatweel(x))
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا من النيتروجين؟,التعليم,الهواء يتكون أساسا من النيتروجين.


In [148]:
test_text = "الـ"
print("does it contain tatweel? ", araby.is_tatweel(test_text))
print("Before Removing Tatweel:")
print(test_text)
print("\nAfter Removing Tatweel:")
print(araby.strip_tatweel(test_text))

does it contain tatweel?  False
Before Removing Tatweel:
الـ

After Removing Tatweel:
ال


### Stop Words

In [149]:
stop_words = set(stopwords.words('arabic'))

# count of stopword tokens per Answer, then get total
stopwords_in_answers = df['Answer'].apply(lambda x: sum(1 for word in word_tokenize(x) if word in stop_words)).sum()
stopwords_in_questions = df['QuestionText'].apply(lambda x: sum(1 for word in word_tokenize(x) if word in stop_words)).sum()

print(f"Total stopword tokens in [Answer] = {stopwords_in_answers}")
print(f"Total stopword tokens in [QuestionText] = {stopwords_in_questions}")

Total stopword tokens in [Answer] = 13634
Total stopword tokens in [QuestionText] = 7358


In [150]:
# Removing stop words:
def remove_ar_stopwords(text):
    filtered_word = [word for word in word_tokenize(text) if word not in stop_words]
    return " ".join(filtered_word)

df['QuestionText'] = df['QuestionText'].apply(remove_ar_stopwords)
df['Answer'] = df['Answer'].apply(remove_ar_stopwords)

df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...
1,أليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر أهم المنتجات الزراعية مصر، ويعد ال...
2,أتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق .
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة .
4,أيتكون الهواء أساسا النيتروجين؟,التعليم,الهواء يتكون أساسا النيتروجين .


### Normalizing the "Hamza":

In [151]:
df['QuestionText'] = df['QuestionText'].apply(lambda x: araby.normalize_hamza(x, method='tasheel')) # the 'tasheel' method changes any hamza to ي just like how when we root words in arabic some of tehm either get converted into ي or و
df['Answer'] = df['Answer'].apply(lambda x: araby.normalize_hamza(x, method='tasheel'))
df.head()

,QuestionText,Category,Answer
0,ايهما افضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...
1,اليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر اهم المنتجات الزراعية مصر، ويعد ال...
2,اتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق .
3,اتعرف البكتيريا بانها كاينات حية دقيقة؟,التعليم,البكتيريا تعرف بانها كاينات حية دقيقة .
4,ايتكون الهواء اساسا النيتروجين؟,التعليم,الهواء يتكون اساسا النيتروجين .


### Harakat

In [152]:
# Check if there are harakat in the dataset
def has_harakat(text):
    return any(char in araby.HARAKAT for char in text)

print("Count of Harakat:")
print(f"num of harakat in [QuestionText] = {df['QuestionText'].apply(has_harakat).sum()}")
print(f"num of harakat in [Answer] = {df['Answer'].apply(has_harakat).sum()}")

Count of Harakat:
num of harakat in [QuestionText] = 0
num of harakat in [Answer] = 0


there are no harakat in the dataset, so we can skip this step.

In [153]:
# check for none alphanumeric characters in the dataset
def has_non_alphanumeric(text):
    return any(not char.isalnum() and not char.isspace() for char in text)
print("Count of Non-Alphanumeric Characters:")
print(f"num of non-alphanumeric characters in [QuestionText] = {df['QuestionText'].apply(has_non_alphanumeric).sum()}")
print(f"num of non-alphanumeric characters in [Answer] = {df['Answer'].apply(has_non_alphanumeric).sum()}")

Count of Non-Alphanumeric Characters:
num of non-alphanumeric characters in [QuestionText] = 5005
num of non-alphanumeric characters in [Answer] = 4891


### Checking for Shadda

In [154]:
# check for shadda in the dataset
def has_special_symbols(text):
    return any(char in araby.SHADDA for char in text)
print("Count of Special Symbols:")
print(f"num of special symbols in [QuestionText] = {df['QuestionText'].apply(has_special_symbols).sum()}")
print(f"num of special symbols in [Answer] = {df['Answer'].apply(has_special_symbols).sum()}")

Count of Special Symbols:
num of special symbols in [QuestionText] = 0
num of special symbols in [Answer] = 0


### Removing Punctuation Marks

In [155]:
# removing punctuation marks
def remove_punctuation(text):
    return ''.join(char for char in text if char.isalnum() or char.isspace())

df['QuestionText'] = df['QuestionText'].apply(remove_punctuation)
df['Answer'] = df['Answer'].apply(remove_punctuation)
df.head()

,QuestionText,Category,Answer
0,ايهما افضل الدراسة السابق الوقت الحالي,التعليم,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...
1,اليس القطن عماد الثروة مصر,الاقتصاد والعمل,القطن يعتبر اهم المنتجات الزراعية مصر ويعد الا...
2,اتصعد الشمس الشرق,التعليم,الشمس تصعد الشرق
3,اتعرف البكتيريا بانها كاينات حية دقيقة,التعليم,البكتيريا تعرف بانها كاينات حية دقيقة
4,ايتكون الهواء اساسا النيتروجين,التعليم,الهواء يتكون اساسا النيتروجين


### Removing Symbols

In [156]:
print("Unique non-letter and non-numbercharacters in [QuestionText]:")
unique_non_letters_questions = set(char for text in df['Answer'] for char in text if not (char in araby.LETTERS or (char.isascii() and char.isalpha()) or char.isspace()))
print(unique_non_letters_questions)

Unique non-letter and non-numbercharacters in [QuestionText]:
{'6', '٢', 'é', '8', '4', '٩', '2', '²', '٦', 'ﷺ', '³', '5', '3', '0', '9', 'π', '1', '7'}


In [157]:
# count of non-letter characters per Answer, then get total
print("Count of Non-Letter Characters:")
def count_non_letters(text):
    return sum(1 for char in text if not (char in araby.LETTERS or (char.isascii() and char.isalpha()) or char.isspace() or char.isdigit()))
non_letters_in_answers = df['Answer'].apply(count_non_letters).sum()
non_letters_in_questions = df['QuestionText'].apply(count_non_letters).sum()
print(f"num of non-letter characters in [QuestionText] = {non_letters_in_questions}")
print(f"num of non-letter characters in [Answer] = {non_letters_in_answers}")

Count of Non-Letter Characters:
num of non-letter characters in [QuestionText] = 4
num of non-letter characters in [Answer] = 16


In [158]:
# remove non-letter characters (keep Arabic letters + English letters + spaces)
def remove_non_letters(text):
    cleaned = ''.join(
        char for char in text
        if char in araby.LETTERS or (char.isascii() and char.isalpha()) or char.isspace() or char.isdigit()
    )
    return ' '.join(cleaned.split())  # normalize extra spaces

df['QuestionText'] = df['QuestionText'].apply(remove_non_letters)
df['Answer'] = df['Answer'].apply(remove_non_letters)

df.head()

,QuestionText,Category,Answer
0,ايهما افضل الدراسة السابق الوقت الحالي,التعليم,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...
1,اليس القطن عماد الثروة مصر,الاقتصاد والعمل,القطن يعتبر اهم المنتجات الزراعية مصر ويعد الا...
2,اتصعد الشمس الشرق,التعليم,الشمس تصعد الشرق
3,اتعرف البكتيريا بانها كاينات حية دقيقة,التعليم,البكتيريا تعرف بانها كاينات حية دقيقة
4,ايتكون الهواء اساسا النيتروجين,التعليم,الهواء يتكون اساسا النيتروجين


### Checking for English Characters

In [159]:
# check if there are englsih characters in the dataset
def contains_english(text):
    return any(char.isalpha() and char.isascii() for char in text)

contains_english_in_question = df['QuestionText'].apply(contains_english)
contains_english_in_answer = df['Answer'].apply(contains_english)

print(f"Number of questions containing English characters: {contains_english_in_question.sum()}")
print(f"Number of answers containing English characters: {contains_english_in_answer.sum()}")

Number of questions containing English characters: 22
Number of answers containing English characters: 133


In [160]:
df['QuestionText'][contains_english_in_question]

163     استنتج القاعدة العامة لسلسلة فيبوناتشي عندما n...
164     استنتج العلاقة الكتلة والتسارع استنادا القانون...
232     التطبيقات التالية الاكثر امانا لحماية البيانات...
551     الاجهزة التالية الافضل للالعاب الالكترونية Pla...
761                             يعني شبكة الجيل الثالث 3G
823         احسب القيمة المجهولة المعادلة التالية 2x 5 15
1482                     لماذا يوجد خط حرفي J وF الكيبورد
1491                              لماذا يوجد خط حرفي J وF
1604         لماذا اعداد فيلم افاتار Avatar اتسم بالغرابة
1860                 لماذا لوك Outlook افضل بريد الكتروني
1907      لماذا تعد خدمة نتفليكس Netflix جاذبة للمستخدمين
1938           لماذا مضروب Factorial الرقم صفر يساوي واحد
2633        اشرح كيفية عمل نظام تحديد المواقع العالمي GPS
2756             تاسست شركة IBM تاثيرها صناعة التكنولوجيا
2929                   عدد الشركات المدرجة بورصة WSE 2009
2988                        تم تجريب الطايرة F15 لاول مرة
3200                   الرييس التنفيذي الحالي لشركة Apple
3202          

In [161]:
# check if é is in the dataset
def contains_e_acute(text):
    return 'é' in text
contains_e_acute_in_question = df['QuestionText'].apply(contains_e_acute)
contains_e_acute_in_answer = df['Answer'].apply(contains_e_acute)
print(f"Number of questions containing 'é': {contains_e_acute_in_question.sum()}")
print(f"Number of answers containing 'é': {contains_e_acute_in_answer.sum()}")

Number of questions containing 'é': 0
Number of answers containing 'é': 0


### Saving the Preprocessed Data using PyArabic Library


In [162]:
df.to_csv('pyarabic_preprocessed_AAFAQ_Dataset.csv', index=False)

 # Preprocessing the AAFAQ Dataset using Regular Expressions

In [165]:
df = pd.read_csv('AAFAQ_Dataset.csv')
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


In [166]:
import re
import pandas as pd
from nltk.corpus import stopwords


# Define Regex Patterns
# Arabic Diacritics (Tashkeel): Range U+064B to U+0652
tashkeel_pattern = re.compile(r'[\u064B-\u0652]')

# Tatweel: U+0640
tatweel_pattern = re.compile(r'\u0640')

# Normalizing Hamza to 'ي'
# Matches: أ, إ, آ, ء, ئ, ؤ
hamza_pattern = re.compile(r'[أإآءئؤ]')

# Punctuation & Symbols: Matches anything NOT a letter, number, or space
# Note: \w in Python 3 includes Arabic letters by default
punctuation_pattern = re.compile(r'[^\w\s]')

# Extra Whitespace
whitespace_pattern = re.compile(r'\s+')

In [172]:
punctuation_pattern = re.compile(r'[^\w\s]')
punctuation_pattern.findall("Hello, world! This is a test. هل هذا يعمل؟")

[',', '!', '.', '؟']

In [168]:
# Setup Stopwords
stop_words = set(stopwords.words('arabic'))

In [169]:
# Define the Cleaning Pipeline
def regex_preprocess(text):
    if not isinstance(text, str):
        return ""
    
    # Remove Tashkeel
    text = tashkeel_pattern.sub('', text)
    
    # Remove Tatweel
    text = tatweel_pattern.sub('', text)
    
    # Normalize Hamza to 'ي' (Tasheel method)
    text = hamza_pattern.sub('ي', text)
    
    # Remove Stopwords using regex splitting (splitting by non-word chars)
    words = re.split(r'\s+', text)
    words = [w for w in words if w not in stop_words]
    text = " ".join(words)
    
    # Remove Punctuation and Symbols
    text = punctuation_pattern.sub('', text)
    
    # Normalize Whitespace (remove double spaces, tabs, etc.)
    text = whitespace_pattern.sub(' ', text).strip()
    
    return text

In [170]:
# Apply to Dataset
df['QuestionText'] = df['QuestionText'].apply(regex_preprocess)
df['Answer'] = df['Answer'].apply(regex_preprocess)

# 5. Review Results
print("Regex Preprocessing Complete.")
df.head()

Regex Preprocessing Complete.


,QuestionText,Category,Answer
0,ييهما يفضل الدراسة السابق يم الوقت الحالي,التعليم,الدراسة الوقت الحالي تعتبر يفضل بسبب توفر التك...
1,يليس القطن عماد الثروة مصر,الاقتصاد والعمل,القطن يعتبر يهم المنتجات الزراعية مصر ويعد الي...
2,يتصعد الشمس الشرق,التعليم,الشمس تصعد الشرق
3,يتعرف البكتيريا بينها كاينات حية دقيقة,التعليم,البكتيريا تعرف بينها كاينات حية دقيقة
4,ييتكون الهواي يساسا النيتروجين,التعليم,الهواي يتكون يساسا النيتروجين


In [171]:
# saving the preprocessed dataset
df.to_csv('regex_preprocessed_AAFAQ_Dataset.csv', index=False)